# Treino do YOLOv8n custom (Sprint 6)

Gera os pesos `models/yolov8n_surgical.pt` consumidos por `src/video/detector.py`. Roda no Google Colab com GPU T4 gratuita. Pipeline auto-contido: baixa CholecSeg8k do Hugging Face, converte mascaras em bounding boxes, treina o YOLOv8n e exporta os artefatos. Idempotente: se o dataset ja existir na pasta configurada, pula direto para o treino.

Contexto da escolha do dataset e do alvo do detector em [docs/arquitetura/decisoes_tecnicas.md](../docs/arquitetura/decisoes_tecnicas.md) (ADR-012).

## 1. Setup

GPU + dependencias + paths. A flag `USE_DRIVE` (default `False`) controla se o dataset convertido fica em `/content/` (efemero) ou em `MyDrive/medica-ia/` (persistente entre sessoes).

In [ ]:
!nvidia-smi
!pip install -q ultralytics==8.3.30 huggingface_hub

USE_DRIVE = False

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DATASET_DIR = '/content/drive/MyDrive/medica-ia/cholecseg8k_yolo'
    EXPORT_DIR = '/content/drive/MyDrive/medica-ia/yolo_runs'
else:
    DATASET_DIR = '/content/cholecseg8k_yolo'
    EXPORT_DIR = '/content/yolo_runs'

RAW_DIR = '/content/cholecseg8k_raw'
DATA_YAML = f'{DATASET_DIR}/data.yaml'
RUNS_DIR = '/content/runs/detect'
RUN_NAME = 'surgical_instruments'

import os
os.makedirs(EXPORT_DIR, exist_ok=True)

print(f'DATASET_DIR={DATASET_DIR}')
print(f'EXPORT_DIR={EXPORT_DIR}')


## 2. Dataset

Pipeline em 3 passos: verifica cache + baixa, extrai + converte mascaras em bboxes, gera `data.yaml`. Tudo idempotente.

### 2.1 Cache + download

Se `DATASET_DIR` ja tem os splits no volume esperado, pula tudo. Caso contrario, baixa `data/CholecSeg8k.zip` (3.1 GB) do HF. A extracao acontece na 2.2.

In [ ]:
def split_is_complete(split: str, min_count: int) -> bool:
    img_dir = os.path.join(DATASET_DIR, 'images', split)
    if not os.path.isdir(img_dir):
        return False
    return len(os.listdir(img_dir)) >= min_count

EXPECTED = {'train': 5000, 'val': 1500, 'test': 700}
NEEDS_PREPARE = not all(split_is_complete(s, n) for s, n in EXPECTED.items())

if NEEDS_PREPARE:
    print(f'Dataset nao encontrado em {DATASET_DIR}, vou baixar.')
    from huggingface_hub import snapshot_download

    snapshot_download(
        repo_id='minwoosun/CholecSeg8k',
        repo_type='dataset',
        local_dir=RAW_DIR,
        allow_patterns=['data/CholecSeg8k.zip'],
    )
    print('Download concluido:')
    !ls -lh {RAW_DIR}/data/CholecSeg8k.zip
else:
    print(f'OK: dataset em cache em {DATASET_DIR}')
    for s, n in EXPECTED.items():
        actual = len(os.listdir(os.path.join(DATASET_DIR, 'images', s)))
        print(f'  {s:>5}: {actual} imagens')


### 2.2 Extracao do zip

Descompacta `CholecSeg8k.zip` (~3 GB) em `RAW_DIR/extracted/`. Usa `zipfile` + `tqdm` para mostrar progresso por arquivo. Tempo esperado no Colab: 3-8 minutos.

In [ ]:
if NEEDS_PREPARE:
    import zipfile
    from pathlib import Path
    from tqdm.auto import tqdm

    zip_path = Path(RAW_DIR) / 'data' / 'CholecSeg8k.zip'
    extracted_root = Path(RAW_DIR) / 'extracted'

    if not extracted_root.exists() or not any(extracted_root.iterdir()):
        extracted_root.mkdir(parents=True, exist_ok=True)
        print(f'Extraindo {zip_path} -> {extracted_root}')
        with zipfile.ZipFile(zip_path) as zf:
            members = zf.namelist()
            for member in tqdm(members, desc='Extraindo', unit='arq'):
                zf.extract(member, extracted_root)
        print(f'Extracao concluida: {len(members)} arquivos em {extracted_root}.')
    else:
        print(f'Zip ja extraido em {extracted_root}, pulando.')
else:
    print('Pulado (cache).')


### 2.3 Conversao mask -> bbox YOLO

Indexa os pares (frame, mascara) extraidos, faz split estratificado 70/20/10 com seed fixa (42), e converte cada mascara grayscale em bounding boxes YOLO. Apenas duas classes alvo: `grasper` (pixel 31) e `l_hook_electrocautery` (pixel 32). Bboxes menores que 100 px sao descartadas. Replica `scripts/convert_cholecseg8k_to_yolo.py` do repo, ajustado para os paths do Colab.

In [ ]:
if NEEDS_PREPARE:
    import shutil
    from pathlib import Path
    import cv2
    import numpy as np
    from tqdm.auto import tqdm

    extracted_root = Path(RAW_DIR) / 'extracted'

    INSTRUMENT_CLASSES = {'grasper': 0, 'l_hook_electrocautery': 1}
    CLASS_PIXEL_VALUE = {'grasper': 31, 'l_hook_electrocautery': 32}
    SPLIT_RATIO = (0.7, 0.2, 0.1)
    SEED = 42
    MIN_BBOX_AREA_PX = 100

    def mask_to_bboxes_by_class(mask_path):
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        if mask is None:
            return {}
        bboxes = {}
        for class_name, pixel_val in CLASS_PIXEL_VALUE.items():
            ys, xs = np.where(mask == pixel_val)
            if len(xs) == 0:
                continue
            x_min, x_max = int(xs.min()), int(xs.max())
            y_min, y_max = int(ys.min()), int(ys.max())
            w, h = x_max - x_min, y_max - y_min
            if w * h < MIN_BBOX_AREA_PX:
                continue
            bboxes[class_name] = (x_min, y_min, w, h)
        return bboxes

    def yolo_format(bbox, img_w, img_h, class_id):
        x, y, w, h = bbox
        cx = (x + w / 2) / img_w
        cy = (y + h / 2) / img_h
        return f'{class_id} {cx:.6f} {cy:.6f} {w / img_w:.6f} {h / img_h:.6f}\n'

    out = Path(DATASET_DIR)
    for split in ('train', 'val', 'test'):
        (out / 'images' / split).mkdir(parents=True, exist_ok=True)
        (out / 'labels' / split).mkdir(parents=True, exist_ok=True)

    print('Indexando pares (frame, mask)...')
    items = []
    for img_path in tqdm(sorted(extracted_root.rglob('frame_*_endo.png')), desc='Indexando', unit='arq'):
        if any(s in img_path.name for s in ('color_mask', 'watershed_mask')):
            continue
        ann_path = img_path.with_name(img_path.name.replace('_endo.png', '_endo_mask.png'))
        if ann_path.exists():
            items.append((img_path, ann_path))

    if not items:
        raise RuntimeError(
            f'Nenhum par (frame, mask) encontrado em {extracted_root}. '
            f'Conferir layout do zip extraido (esperado: subpastas video_XX/video_XX_YY/ com frame_*_endo.png e frame_*_endo_mask.png).'
        )

    print(f'Pares encontrados: {len(items)}')

    rng = np.random.default_rng(seed=SEED)
    indices = list(range(len(items)))
    rng.shuffle(indices)
    n = len(items)
    n_train = int(n * SPLIT_RATIO[0])
    n_val = int(n * SPLIT_RATIO[1])
    splits = {
        'train': indices[:n_train],
        'val': indices[n_train:n_train + n_val],
        'test': indices[n_train + n_val:],
    }

    cls_count = {cid: 0 for cid in INSTRUMENT_CLASSES.values()}
    for split_name, idxs in splits.items():
        for i in tqdm(idxs, desc=f'Convertendo {split_name}', unit='frame'):
            img_path, ann_path = items[i]
            unique_name = f'{img_path.parent.name}_{img_path.name}'
            unique_stem = f'{img_path.parent.name}_{img_path.stem}'
            shutil.copy(img_path, out / 'images' / split_name / unique_name)
            dst_lbl = out / 'labels' / split_name / f'{unique_stem}.txt'

            img = cv2.imread(str(img_path))
            if img is None:
                dst_lbl.touch()
                continue
            h, w = img.shape[:2]

            bboxes = mask_to_bboxes_by_class(ann_path)
            lines = []
            for class_name, bbox in bboxes.items():
                cid = INSTRUMENT_CLASSES[class_name]
                lines.append(yolo_format(bbox, w, h, cid))
                cls_count[cid] += 1
            dst_lbl.write_text(''.join(lines) if lines else '')

    print(f'\nResumo: {n} frames processados')
    for name, cid in sorted(INSTRUMENT_CLASSES.items(), key=lambda kv: kv[1]):
        print(f'  {name} (id {cid}): {cls_count[cid]} bboxes')
    print(f"Splits: train={len(splits['train'])} val={len(splits['val'])} test={len(splits['test'])}")
else:
    print('Pulado (cache).')


### 2.4 data.yaml

Aponta para a raiz do dataset no Colab (sobrescreve o path relativo do repo local).

In [ ]:
import yaml

with open(DATA_YAML, 'w') as f:
    yaml.safe_dump({
        'path': DATASET_DIR,
        'train': 'images/train',
        'val': 'images/val',
        'test': 'images/test',
        'nc': 2,
        'names': {0: 'grasper', 1: 'l_hook_electrocautery'},
    }, f, sort_keys=False)

print(open(DATA_YAML).read())


### 2.5 Validacao + visualizacao

Sanity check: contagem por split e quatro amostras com bbox sobre o frame.

In [ ]:
for split in ['train', 'val', 'test']:
    img_dir = os.path.join(DATASET_DIR, 'images', split)
    lbl_dir = os.path.join(DATASET_DIR, 'labels', split)
    n_imgs = len(os.listdir(img_dir))
    n_nonempty = sum(1 for f in os.listdir(lbl_dir) if os.path.getsize(os.path.join(lbl_dir, f)) > 0)
    print(f'{split:>5}: {n_imgs:>5} imagens ({n_nonempty} com bbox)')


In [ ]:
import random
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches

random.seed(42)

CLASS_COLORS = {0: 'lime', 1: 'magenta'}
CLASS_LABELS = {0: 'grasper', 1: 'l_hook'}

img_dir = os.path.join(DATASET_DIR, 'images', 'train')
lbl_dir = os.path.join(DATASET_DIR, 'labels', 'train')

candidates = [f for f in os.listdir(img_dir)
              if os.path.getsize(os.path.join(lbl_dir, f.replace('.png', '.txt'))) > 0]

if not candidates:
    raise RuntimeError(
        f'Nenhuma imagem com bbox em {lbl_dir}. '
        'A conversao da 2.2 nao gerou labels (ou o cache em DATASET_DIR ficou inconsistente). '
        'Apague DATASET_DIR e rode tudo de novo a partir da 2.1.'
    )

samples = random.sample(candidates, min(4, len(candidates)))
n = len(samples)
rows = (n + 1) // 2
fig, axes = plt.subplots(rows, 2, figsize=(14, 5 * rows))
axes_list = list(axes.flat) if n > 1 else [axes]

for ax, name in zip(axes_list, samples):
    img = Image.open(os.path.join(img_dir, name))
    W, H = img.size
    ax.imshow(img)
    ax.set_title(name, fontsize=9)
    ax.axis('off')

    with open(os.path.join(lbl_dir, name.replace('.png', '.txt'))) as f:
        for line in f:
            parts = line.strip().split()
            cls = int(parts[0])
            xc, yc, w, h = map(float, parts[1:5])
            x0 = (xc - w / 2) * W
            y0 = (yc - h / 2) * H
            ax.add_patch(patches.Rectangle(
                (x0, y0), w * W, h * H,
                linewidth=2, edgecolor=CLASS_COLORS.get(cls, 'cyan'), facecolor='none',
            ))
            ax.text(x0, max(0, y0 - 5), CLASS_LABELS.get(cls, str(cls)),
                    color=CLASS_COLORS.get(cls, 'cyan'), fontsize=9,
                    bbox=dict(facecolor='black', alpha=0.6, pad=2))

for ax in axes_list[n:]:
    ax.axis('off')

plt.tight_layout()
plt.show()


## 3. Treino

Treino na GPU T4 do Colab (`device=0` no `model.train()`). Partimos dos pesos `yolov8n.pt` (pre-treinado em COCO) e fazemos fine-tuning nas duas classes alvo. Escolhemos a variante **nano** porque o app Gradio do projeto roda os pesos finais em CPU local (sem GPU dedicada), entao a inferencia precisa ser leve. O treino em si nao tem essa restricao: usa a GPU do Colab.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

results = model.train(
    data=DATA_YAML,
    epochs=40,
    imgsz=640,
    batch=16,
    device=0,
    name=RUN_NAME,
    patience=10,
    project=RUNS_DIR,
    exist_ok=True,
    verbose=True,
)


**Esperado:** `mAP50 > 0.7` ao fim do treino. Se travar abaixo de 0.3 nas primeiras 5 epocas, revisar splits/labels.

## 4. Avaliacao

Metricas no split de teste + curvas + matriz de confusao + predicoes visuais.

In [ ]:
best_path = f'{RUNS_DIR}/{RUN_NAME}/weights/best.pt'
best_model = YOLO(best_path)

metrics = best_model.val(
    data=DATA_YAML,
    split='test',
    project=RUNS_DIR,
    name=f'{RUN_NAME}_test',
    exist_ok=True,
)

print(f'mAP50:     {metrics.box.map50:.4f}')
print(f'mAP50-95:  {metrics.box.map:.4f}')
print(f'precision: {metrics.box.mp:.4f}')
print(f'recall:    {metrics.box.mr:.4f}')
print()
print('mAP50 por classe:')
for i, name in metrics.names.items():
    print(f'  {name:<25s} {metrics.box.maps[i]:.4f}')


In [ ]:
from IPython.display import Image as IPImage, display

display(IPImage(f'{RUNS_DIR}/{RUN_NAME}/results.png'))
display(IPImage(f'{RUNS_DIR}/{RUN_NAME}_test/confusion_matrix.png'))


In [ ]:
test_img_dir = os.path.join(DATASET_DIR, 'images', 'test')
test_samples = random.sample(os.listdir(test_img_dir), 4)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, name in zip(axes.flat, test_samples):
    img_path = os.path.join(test_img_dir, name)
    pred = best_model(img_path, conf=0.25, verbose=False)[0]
    annotated = pred.plot()
    ax.imshow(annotated[..., ::-1])
    ax.set_title(name, fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.show()


## 5. Export

Copia `best.pt` + graficos para `EXPORT_DIR`. Baixar `best.pt` pela barra lateral do Colab (ou pelo Drive se `USE_DRIVE=True`), colocar em `models/yolov8n_surgical.pt` no repo e ajustar `YOLO_WEIGHTS_PATH` no `.env`.

In [ ]:
import shutil

artifacts = {
    'best.pt': f'{RUNS_DIR}/{RUN_NAME}/weights/best.pt',
    'last.pt': f'{RUNS_DIR}/{RUN_NAME}/weights/last.pt',
    'results.png': f'{RUNS_DIR}/{RUN_NAME}/results.png',
    'results.csv': f'{RUNS_DIR}/{RUN_NAME}/results.csv',
    'confusion_matrix.png': f'{RUNS_DIR}/{RUN_NAME}_test/confusion_matrix.png',
    'val_predictions.jpg': f'{RUNS_DIR}/{RUN_NAME}/val_batch0_pred.jpg',
}

for dst_name, src in artifacts.items():
    if os.path.exists(src):
        dst = os.path.join(EXPORT_DIR, dst_name)
        shutil.copy(src, dst)
        size_mb = os.path.getsize(dst) / (1024 * 1024)
        print(f'OK   {dst_name:<25s} ({size_mb:.2f} MB)')
    else:
        print(f'SKIP {dst_name:<25s}')


## Fontes

- CholecSeg8k: Hong et al. (2020), [arXiv:2012.12463](https://arxiv.org/abs/2012.12463) (CC BY-NC-SA 4.0)
- Ultralytics YOLOv8: [docs.ultralytics.com](https://docs.ultralytics.com/)
- Repo do projeto: [github.com/joalissonborges94/medica-ia-multimodal](https://github.com/joalissonborges94/medica-ia-multimodal)